In [ ]:
import sys
import time
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, from_json, current_timestamp, to_date, 
    year, month
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, 
    FloatType, TimestampType
)

# --- 1. CONFIGURATION (WIDGETS) ---
dbutils.widgets.text("kafka_bootstrap_servers", "0.tcp.ap.ngrok.io:17438", "Kafka Ngrok URL")
dbutils.widgets.text("kafka_topic", "audio_uploads", "Kafka Topic")
dbutils.widgets.text("catalog_name", "fma_catalog", "Unity Catalog Name")

kafka_url = dbutils.widgets.get("kafka_bootstrap_servers")
kafka_topic = dbutils.widgets.get("kafka_topic")
catalog = dbutils.widgets.get("catalog_name")
checkpoint_path = f"dbfs:/checkpoints/{catalog}/bronze_audio"

# --- 2. S3 & UNITY CATALOG SETUP ---
try:
    spark.conf.set("fs.s3a.access.key", dbutils.secrets.get(scope="fma-scope", key="s3-access-key"))
    spark.conf.set("fs.s3a.secret.key", dbutils.secrets.get(scope="fma-scope", key="s3-secret-key"))
    spark.conf.set("fs.s3a.endpoint", "s3.eu-north-1.amazonaws.com")
    print("S3 Credentials Loaded.")
except:
    print("Secret scope not found. Ensure you ran: databricks secrets create-scope --scope fma-scope")

# Initialize Database
spark.sql(f"CREATE DATABASE IF NOT EXISTS {catalog}.bronze")
spark.sql(f"USE {catalog}.bronze")

# --- 3. DEFINE SCHEMA ---
kafka_message_schema = StructType([
    StructField("track_id", IntegerType(), False),
    StructField("metadata", StructType([
        StructField("title", StringType(), True),
        StructField("duration", FloatType(), True),
        StructField("artist_name", StringType(), True),
        StructField("genre_top", StringType(), True)
    ]), True),
    StructField("s3_uri", StringType(), False),
    StructField("timestamp", StringType(), True) 
])

# --- 4. READ FROM KAFKA ---
kafka_stream_df = (spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", kafka_url)
    .option("subscribe", kafka_topic)
    .option("startingOffsets", "latest")
    .load())

# --- 5. TRANSFORM & PARSE ---
bronze_df = (kafka_stream_df
    .selectExpr("CAST(value AS STRING) as json_payload", "timestamp as kafka_time")
    .select(from_json(col("json_payload"), kafka_message_schema).alias("data"), "kafka_time")
    .select(
        "data.track_id",
        "data.metadata.*",
        "data.s3_uri",
        col("data.timestamp").cast(TimestampType()).alias("event_time"),
        "kafka_time",
        current_timestamp().alias("ingested_at")
    )
    .withColumn("event_date", to_date(col("event_time")))
)

# --- 6. START STREAMING TO DELTA ---
table_name = f"{catalog}.bronze.audio_events"

print(f"Starting Stream: Kafka -> {table_name}")

query = (bronze_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .partitionBy("event_date")
    .toTable(table_name))

# --- 7. MONITORING LOOP ---
time.sleep(10)
if query.isActive:
    print(f"Stream is ACTIVE. Query ID: {query.id}")
    print(f"Status: {query.status['message']}")